In [149]:
# ----------------------------------------------------
# STEP 1 — Import Libraries
# ----------------------------------------------------
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import pickle

print("Libraries imported successfully!")


Libraries imported successfully!


In [134]:
# STEP 2 — Load Dataset

df = pd.read_csv("Telco-Customer-Churn.csv")
# First 5 rows display
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [135]:
# Check dataset info

print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [136]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [137]:

# Check missing values

df.isnull().sum()
print("Selected columns:", df.columns.tolist())

Selected columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [138]:
# Convert TotalCharges to numeric 

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")


In [139]:

# Drop customerID 

df.drop("customerID", axis=1, inplace=True)

print("Basic cleaning done!")

Basic cleaning done!


In [140]:
# STEP 4 — Select ONLY 8 FEATURES


selected_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "Contract",
    "PaymentMethod",
    "InternetService",
    "OnlineSecurity",
    "TechSupport"
]

In [141]:
# Create new dataframe with selected features

df = df[selected_features + ["Churn"]]

print("Selected features applied!")
df.head()

Selected features applied!


,tenure,MonthlyCharges,TotalCharges,Contract,PaymentMethod,InternetService,OnlineSecurity,TechSupport,Churn
0,1,29.85,29.85,Month-to-month,Electronic check,DSL,No,No,No
1,34,56.95,1889.50,One year,Mailed check,DSL,Yes,No,No
2,2,53.85,108.15,Month-to-month,Mailed check,DSL,Yes,No,Yes
3,45,42.30,1840.75,One year,Bank transfer (automatic),DSL,Yes,Yes,No
4,2,70.70,151.65,Month-to-month,Electronic check,Fiber optic,No,No,Yes


In [142]:
# STEP 5 — Identify categorical & numeric columns

# Categorical columns
categorical_cols = df.select_dtypes(include=["object"]).columns

# Numerical columns
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns

print("Categorical Columns:")
print(categorical_cols)

print("\nNumerical Columns:")
print(numeric_cols)
print("Model trained successfully!")

Categorical Columns:
Index(['Contract', 'PaymentMethod', 'InternetService', 'OnlineSecurity',
       'TechSupport', 'Churn'],
      dtype='str')

Numerical Columns:
Index(['tenure', 'MonthlyCharges', 'TotalCharges'], dtype='str')
Model trained successfully!


In [143]:
# STEP 6 — Encode categorical columns

le = LabelEncoder()

# Apply Label Encoding to all categorical columns
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

print("Categorical columns encoded!")
df.head()


Categorical columns encoded!


,tenure,MonthlyCharges,TotalCharges,Contract,PaymentMethod,InternetService,OnlineSecurity,TechSupport,Churn
0,1,29.85,29.85,0,2,0,0,0,0
1,34,56.95,1889.50,1,3,0,2,0,0
2,2,53.85,108.15,0,3,0,2,0,1
3,45,42.30,1840.75,1,0,0,2,2,0
4,2,70.70,151.65,0,2,1,0,0,1


In [144]:
# STEP 7 — Train/Test Split

# Separate features (X) and target (y)
X = df.drop("Churn", axis=1)
y = df["Churn"]

# Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train/Test split done!")
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

Train/Test split done!
X_train shape: (5634, 8)
X_test shape: (1409, 8)


In [145]:

# STEP 8 — Scale numerical columns

scaler = StandardScaler()

# Apply scaling only on numerical columns
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print("Feature scaling done!")

Feature scaling done!


In [146]:
# STEP 9 — Train Model

# Initialize model
model = RandomForestClassifier(random_state=42)

# Train model
model.fit(X_train, y_train)

print("Model training completed!")

Model training completed!


In [147]:
# STEP 10 — Evaluate Model

# Make predictions
y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Detailed report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.78708303761533

Classification Report:

              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1036
           1       0.62      0.49      0.55       373

    accuracy                           0.79      1409
   macro avg       0.73      0.69      0.71      1409
weighted avg       0.78      0.79      0.78      1409



In [148]:
# STEP 11 — Save model, scaler, encoder

# Save model
pickle.dump(model, open("streaming_churn_model_1.pkl","wb"))

# Save scaler
pickle.dump(scaler, open("scaler_1.pkl","wb"))

# Save encoders 
pickle.dump(le, open("encoder_1.pkl","wb"))

print("Model, Scaler, and Encoders saved successfully!")

Model, Scaler, and Encoders saved successfully!
